# CompostIQ — compost stage classifier

This notebook builds a model that reads a bin's sensor history and says which of **our five
composting stages** the bin is in:

| id | Stage | Simulator envelope (°C) |
|---|---|---|
| 0 | Early Mesophilic | 28–45 |
| 1 | Active Thermophilic | 45–60 |
| 2 | Peak Decomposition | 57–65 |
| 3 | Cooling | 30–60 |
| 4 | Maturation | 28–32 |

It uses the same pile physics as the device emulator (`emulator/model.py`, driven by
`simulator/composting_stages.py`) to generate labelled batches. It then engineers features
backed by the compost literature, compares which sensors are worth having, and trains four
models: a literature rule set, a decision tree small enough for the ESP32, gradient boosting
and a PyTorch GRU. Finally it stress-tests them on bins that don't look like the training data.

**How to run:** from the repo root, `pip install -r ml/requirements.txt`, then open this
notebook and *Run All*. It takes about 7 minutes on a laptop CPU. Outputs (figures,
the exported tree, models) go to `ml/outputs/`.

> **Read this first: all labels here come from our own simulator.** A model trained on them
> can only be as right as the simulator's stage envelopes. Section 1 checks those envelopes
> against the literature, and the conclusions list what real pilot data would need to confirm.

## Summary of results

Scores are macro-F1 (1.0 = perfect) on 30 simulated test batches the models never saw. "Worst
stress test" is the lowest score across hot season, neglected, small basket and noisy sensors
(section 9).

| Model | Runs on | Test | Worst stress test |
|---|---|---|---|
| Literature rules (no training) | anywhere | 0.764 | 0.034 (small basket) |
| Decision tree, °C features | ESP32 | 0.898 | 0.013 (small basket) |
| Gradient boosting + stage filter, °C features | server | 0.903 | 0.013 (small basket) |
| GRU (PyTorch), raw readings | server | **0.933** | 0.293 (small basket) |
| **Gradient boosting + filter, scale-free features, varied bins** | server | **0.932** | **0.891** (hot season) |
| Decision tree, scale-free features, varied bins (exported) | ESP32 | 0.827 | 0.662 (neglected) |

1. **Recommended model: scale-free features, trained on varied bin sizes, with the stage filter.** It matches the GRU on normal bins and is the only model that stays at or above 0.89 in every stress test. Any model that reads absolute °C collapses when a bin never gets hot.
2. **Moisture is the most valuable extra sensor** (0.882 → 0.935). A second DS18B20 for ambient adds little on standard bins, but the robust features are built on it.
3. **The simulator's gas readings give the label away.** They jump at every stage change, and a model using them scores 1.000. Until that is fixed we can't judge whether a gas sensor helps.
4. **51 % of "Cooling" readings come from piles drier than 30 %** (the emulator's dry-stall rule). That's a fault, not a stage, and it causes most of the remaining errors (Maturation predicted as Cooling).
5. **A random split flatters the score:** 0.983 vs. 0.941 when split by batch. With three pilot bins, a random split promises 0.989 while unseen batches get 0.892.
6. **The stage filter steadies the output.** It cuts about 8 stage changes per batch down to about 4 (the true number), at the cost of 1–5 h median lag and no loss of accuracy.
7. **The SmartBox Peltier data doesn't transfer** (0.31 and 0.16 macro-F1), because a heated box never cools to ambient. Don't reuse that model for our bins.

## 1. What the research says

### 1.1 The phases of aerobic composting

| Phase | Core temperature | What is happening | Typical length |
|---|---|---|---|
| Mesophilic (start) | ambient → about 40–45 °C | Easy sugars are eaten first; oxygen demand and heat output peak early | 1–2 days [[Cornell CWMI]](https://compost.css.cornell.edu/microorg.html) [[CCE Tompkins]](https://ccetompkins.org/resources/compost-testing-for-finished-compost) |
| Thermophilic | above about 40–45 °C; keep below 65 °C (microbes die or go dormant above 71 °C) | Fastest breakdown; pathogens die at ≥ 55 °C; NH₃ emission rises exponentially with temperature | days to weeks: days 3–14 in a home pile [[CCE Tompkins]](https://ccetompkins.org/resources/compost-testing-for-finished-compost), more than 7 weeks above 50 °C in a large windrow [[Cabañas-Vargas 2005]](https://www.revista.ingenieria.uady.mx/volumen9/assessing.pdf) [[LSU AgCenter]](https://agsci.oregonstate.edu/sites/agscid7/files/snic/basic-principles-of-composting-lsu.pdf) [[Pagans 2006]](https://ddd.uab.cat/pub/artpub/2006/163578/chemosphere_a2006m3v62n9p1534.pdf) |
| Cooling | falling back through the mesophilic band | O₂ demand, CO₂ and NH₃ fall; mesophiles return | days to weeks [[Cornell CWMI]](https://compost.css.cornell.edu/microorg.html) |
| Curing / maturation | near ambient, **no reheating** | Humus forms; NH₄⁺ turns into NO₃⁻ | 4–8 weeks even in-vessel, months in piles [[Lim 2017]](https://www.aidic.it/cet/17/61/173.pdf) |

### 1.2 Thresholds a sensor can actually see

- **Pathogen kill is a time × temperature rule.**
  - US EPA PFRP: ≥ 55 °C for 3 days (in-vessel or aerated static pile), or ≥ 55 °C for 15 days with 5 turnings (windrow) [[40 CFR 503 App. B]](https://www.law.cornell.edu/cfr/text/40/appendix-B_to_part_503).
  - EU 2019/1009: 70 °C ≥ 3 d, 65 °C ≥ 5 d, 60 °C ≥ 7 d or 55 °C ≥ 14 d [[EU 2019/1009 Annex II]](https://www.legislation.gov.uk/eur/2019/1009/annex/II/2020-01-31).
  - These rules are why the notebook counts **hours at ≥ 55 °C**.
- **Moisture:** 40–60 % is optimal. Above 60 % water fills the pores and the pile goes anaerobic, and activity stops below 15 % [[Cornell moisture]](https://www.compost.css.cornell.edu/monitor/monitormoisture.html) [[LSU AgCenter]](https://agsci.oregonstate.edu/sites/agscid7/files/snic/basic-principles-of-composting-lsu.pdf).
- **Oxygen:** ≥ 5 % in the pores for organisms to survive; below 10 % parts of the pile go anaerobic [[Cornell FS4]](https://www.compost.css.cornell.edu/Factsheets/FS4.html).
- **Maturity:** "back to ambient temperature" is a recognised maturity criterion [[Huang 2017]](https://pmc.ncbi.nlm.nih.gov/articles/PMC5467826/). Mature compost no longer self-heats (Dewar Rottegrad V means Tmax 20–30 °C) [[Cabañas-Vargas 2005]](https://www.revista.ingenieria.uady.mx/volumen9/assessing.pdf). These are the only maturity tests a temperature probe can stand in for. C:N, germination index and NH₄/NO₃ all need a lab.

### 1.3 What this means for our classifier

1. **"Peak Decomposition" is not a separate phase in the literature.** It is the hot end of the thermophilic phase (≥ 55 °C sustained). Our stages 1 and 2 overlap at 57–60 °C, so only *history* can separate them: how long the bin has been hot, and whether it is still climbing. Section 10 tests a 4-stage version that merges them.
2. **Temperature bands don't transfer between bins.**
   - A sensor study found phase temperatures didn't match the textbook bands under non-standard ambient conditions [[Ma 2025]](https://www.cetjournal.it/cet/25/122/004.pdf).
   - An Indonesian Takakura basket peaked at only **35 °C** and was mature by day 22 [[Nurita 2023]](https://ojs.poltekkesbengkulu.ac.id/index.php/sanpro/article/download/383/236).
   - Small loads usually can't reach sanitising temperatures [[Lim 2017]](https://www.aidic.it/cet/17/61/173.pdf).
   - → We build features **relative to ambient and to the bin's own peak**, and stress-test on "small basket" bins.
3. **Bokashi is not composting in this sense.** It is lactic fermentation in a closed bucket (leachate pH 3.7–4.9) [[FEMS Microbes 2026]](https://doi.org/10.1093/femsmc/xtag018), and a smart bokashi bin peaked at only 38–39 °C [[Lew 2021]](https://pmc.ncbi.nlm.nih.gov/articles/PMC8073414/). The five stages don't apply to it, so **this notebook covers aerobic bins only** (Takakura, aerated piles).
4. **MQ-135 is a qualitative trend, not a gas concentration.**
   - It responds to NH₃, NOx, alcohol, benzene, smoke and CO₂.
   - Its rated range is −10 to 45 °C at RH below 95 %, it needs more than 48 h of preheat and per-unit calibration, and it cannot measure O₂ [[Hanwei datasheet]](https://cdn.shopify.com/s/files/1/2132/9029/files/MQ-135.pdf) [[Winsen manual]](https://www.winsen-sensor.com/manual/mq135.html).
   - A thermophilic headspace is at or beyond those limits.
5. **The ML literature mostly evaluates in a way that leaks.**
   - Ma 2025 reports R² > 0.99 from a random 80:20 split of 5-minute records inside each run.
   - Random cross-validation underestimates error on dependent data [[Roberts 2017]](https://www.biometrie.uni-freiburg.de/mitarbeiter/dormann/roberts-et-al-2017-ecography.pdf), and temporal leakage is a top cause of irreproducible ML [[Kapoor & Narayanan 2023]](https://ar5iv.labs.arxiv.org/html/2207.07048).
   - → **We split by batch, use only trailing (past-only) windows, and measure how much a random split flatters the score.**
6. We found **no LSTM/GRU study** of compost phase classification, so the GRU in section 8 is exploratory.

### 1.4 Our simulator's envelopes against the literature

| Stage | Verdict | Note |
|---|---|---|
| 0 Early Mesophilic | Temperature OK; O₂ low | Literature: O₂ starts near 21 %, so 12–20 % fits better than 8–15 % |
| 1 Active Thermophilic | OK | Days are plausible in-vessel; unsupported for baskets |
| 2 Peak Decomposition | **Not a recognised phase**; O₂ 2–5 % is below the 5 % survival floor | Merge into thermophilic, or define it as "≥ 55 °C sustained" |
| 3 Cooling | Temperature OK; needs slope to tell apart from stage 1 | Literature: weeks in windrows |
| 4 Maturation | OK | Better expressed as "ΔT to ambient ≈ 0, no reheating" |
| all | O₂ + CO₂ don't add up | With a respiratory quotient ≈ 1, O₂ + CO₂ stays near 21 %; stage 2 sums to only 8–15 % ([Jackson & Line 1998](https://eprints.utas.edu.au/11434/) use the CO₂/O₂ quotient as a diagnostic) |

## 2. Setup
The notebook finds the repo root on its own, so it runs from `ml/` or from the repo root.

In [ ]:
import importlib.util
import json
import math
import os
import random
import sys
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, balanced_accuracy_score,
                             confusion_matrix, f1_score)
from sklearn.model_selection import GridSearchCV, GroupKFold, KFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.precision", 3)
pd.set_option("display.width", 140)


def find_repo(start):
    for folder in [start, *start.parents]:
        if (folder / "emulator" / "model.py").exists() and (folder / "simulator" / "composting_stages.py").exists():
            return folder
    raise FileNotFoundError("Open this notebook from inside the compost_iot_system repo")


REPO = find_repo(Path.cwd().resolve())
OUT = REPO / "ml" / "outputs"
OUT.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(REPO / "simulator"))
from composting_stages import STAGES                    # the stage table the simulator and emulator share
from mock_data import _mq135_response as mq135_response  # the repo's own MQ-135 composite model

# load emulator/model.py by path so its generic module name can't clash with anything else
_spec = importlib.util.spec_from_file_location("compost_model", REPO / "emulator" / "model.py")
compost_model = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(compost_model)
CompostModel = compost_model.CompostModel

STAGE_IDS = [s["id"] for s in STAGES]
STAGE_NAMES = [s["name"] for s in STAGES]
LAST_STAGE = STAGE_IDS[-1]

# chart styling: categorical slots 1-5 (validated for colour-vision deficiency, adjacent pairs)
STAGE_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
SERIES = STAGE_COLORS                 # same fixed order for any other categorical series
INK, INK_2, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#8a8984", "#e4e3df", "#fcfcfb"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150, "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": MUTED,
    "axes.labelcolor": INK_2, "xtick.color": INK_2, "ytick.color": INK_2, "text.color": INK,
    "axes.titleweight": "bold", "axes.titlesize": 11, "font.size": 9.5, "lines.linewidth": 1.6,
    "legend.frameon": False,
})


def savefig(fig, name):
    fig.savefig(OUT / name, bbox_inches="tight")


print("repo:", REPO)
print("outputs:", OUT)
pd.DataFrame(STAGES).set_index("id")

: 

## 3. Simulating labelled batches

We drive `CompostModel` (the same class the device emulator uses) one 30-minute step at a time,
which matches the simulator's own `SAMPLE_INTERVAL_MINUTES = 30`. The model knows the true stage,
and that becomes the label. Around it we add the parts the emulator leaves out, each grounded in
section 1:

| Added layer | Why | Range used |
|---|---|---|
| Ambient day/night swing | Tropical diurnal cycle at the Mataram pilot (ambient about 24–32 °C) | base 25–31 °C, ±2.5–4.5 °C, peak mid-afternoon |
| Ambient leaking into the core | Small bins aren't perfectly insulated | 10–35 % of the outdoor swing reaches the probe |
| Owner behaviour | Real bins are turned and watered irregularly, or not at all | turn every 2/3/5/7 days or never; water when moisture < 42/45/48 % or never |
| Bin size (`heat_scale`) | Household baskets may only reach about 35 °C | 1.0 = simulator as-is; 0.2–0.5 = small basket |
| Second probe for ambient | Planned hardware: a second DS18B20 on the same 1-Wire bus | ±0.3 °C noise |
| MQ-135 composite | The repo's own `mock_data._mq135_response` (CO₂ + NH₃ + noise) | as in the mock dataset |
| Missed readings | Same 2 % chance as `simulator/generator.py` | — |

In [ ]:
STEP_H = 0.5                 # one reading every 30 minutes
MISSED_READING_CHANCE = 0.02
SENSOR_COLS = ["temperature_c", "moisture_pct", "ambient_c", "o2_pct", "co2_pct", "nh3_relative", "mq135"]


def simulate_batch(seed, ambient=(25.0, 31.0), heat_scale=(1.0, 1.0), noise_scale=1.0,
                   care="random", max_days=90):
    """One batch from fresh feedstock until it's cured (or max_days). Returns (readings, settings)."""
    r = random.Random(seed)
    amb_base = r.uniform(*ambient)
    amb_swing = r.uniform(2.5, 4.5)
    leak = r.uniform(0.10, 0.35)
    heat = r.uniform(*heat_scale)
    if care == "neglected":
        turn_every, water_below = None, None
    else:
        turn_every = r.choice([None, 2, 3, 5, 7])
        water_below = r.choice([None, 42.0, 45.0, 48.0])

    pile = CompostModel(rng=random.Random(seed + 1), ambient_temp_c=amb_base)
    dt_days = STEP_H / 24
    next_turn = turn_every if turn_every else math.inf
    extra_noise = max(0.0, noise_scale - 1.0)
    rows = []
    for k in range(int(max_days * 24 / STEP_H)):
        if pile.cured:
            break
        hours = k * STEP_H
        hour_of_day = hours % 24
        amb = amb_base + amb_swing * math.sin(2 * math.pi * (hour_of_day - 9) / 24)

        # the owner checks the bin once a day at 08:00
        if hour_of_day == 8 and water_below is not None and pile.moisture < water_below:
            pile.add_water()
        if hours / 24 >= next_turn and pile.stage_id < LAST_STAGE:
            pile.turn()
            next_turn += turn_every

        reading = pile.read()
        if r.random() >= MISSED_READING_CHANCE:
            core = amb_base + heat * (reading["temperature_c"] - amb_base) + leak * (amb - amb_base)
            rows.append({
                "elapsed_h": hours,
                "temperature_c": round(core + r.gauss(0, 0.5 * extra_noise), 2),
                "moisture_pct": round(reading["moisture_pct"] + r.gauss(0, 1.0 * extra_noise), 2),
                "ambient_c": round(amb + r.gauss(0, 0.3), 2),
                "o2_pct": reading["o2_pct"],
                "co2_pct": reading["co2_pct"],
                "nh3_relative": reading["nh3_relative"],
                "mq135": round(mq135_response(reading["co2_pct"], reading["nh3_relative"], r), 1),
                "stage_id": pile.stage_id,
            })
        pile.step(dt_days)

    settings = {"ambient_base": amb_base, "ambient_swing": amb_swing, "core_leak": leak, "heat_scale": heat,
                "turn_every_days": turn_every, "water_below_pct": water_below,
                "days": rows[-1]["elapsed_h"] / 24 if rows else 0, "cured": pile.cured}
    return pd.DataFrame(rows), settings


def simulate_many(name, n, seed0, **scenario):
    frames, settings = [], []
    for i in range(n):
        df, cfg = simulate_batch(seed0 + i, **scenario)
        batch_id = f"{name}-{i:03d}"
        frames.append(df.assign(batch_id=batch_id))
        settings.append({"batch_id": batch_id, **cfg})
    return pd.concat(frames, ignore_index=True), pd.DataFrame(settings)


SCENARIOS = {
    # name           batches, first seed, and what's different about them
    "train":          dict(n=100, seed0=0),
    "test":           dict(n=30, seed0=10_000),
    "hot_season":     dict(n=20, seed0=20_000, ambient=(30.0, 34.0)),
    "neglected":      dict(n=20, seed0=30_000, care="neglected"),
    "small_basket":   dict(n=20, seed0=40_000, heat_scale=(0.2, 0.5)),
    "noisy_sensors":  dict(n=20, seed0=50_000, noise_scale=2.5),
}

t0 = time.time()
raw, batch_info = {}, {}
for name, spec in SCENARIOS.items():
    raw[name], batch_info[name] = simulate_many(name, **spec)
print(f"simulated {sum(len(v) for v in raw.values()):,} readings in {time.time() - t0:.1f}s")

pd.DataFrame({
    name: {
        "batches": info.batch_id.nunique(),
        "readings": len(raw[name]),
        "median days": info.days.median(),
        "cured within 90 d": f"{info.cured.mean():.0%}",
        "peak °C (median)": raw[name].groupby("batch_id").temperature_c.max().median(),
    }
    for name, info in batch_info.items()
}).T

In [ ]:
# how much of each batch is spent in each stage (training set)
share = raw["train"].stage_id.value_counts(normalize=True).sort_index()
days = (raw["train"].groupby(["batch_id", "stage_id"]).size() * STEP_H / 24).groupby("stage_id").median()

fig, ax = plt.subplots(figsize=(8, 2.8))
bars = ax.barh(STAGE_NAMES[::-1], share.values[::-1] * 100, color=SERIES[0], height=0.6)
for bar, d, s in zip(bars, days.values[::-1], share.values[::-1]):
    ax.text(bar.get_width() + 0.8, bar.get_y() + bar.get_height() / 2,
            f"{s:.0%}  ·  median {d:.1f} days", va="center", color=INK_2, fontsize=8.5)
ax.set_xlabel("% of readings")
ax.set_xlim(0, 75)
ax.set_title("Maturation dominates the data, so we score with macro-F1, not accuracy")
ax.grid(axis="y", visible=False)
savefig(fig, "01_stage_share.png")
plt.show()

In [ ]:
def shade_stages(ax, hours, stages):
    """Colour the background of a time axis by stage."""
    days = np.asarray(hours) / 24
    stages = np.asarray(stages)
    start = 0
    for i in range(1, len(stages) + 1):
        if i == len(stages) or stages[i] != stages[start]:
            ax.axvspan(days[start], days[min(i, len(days) - 1)], color=STAGE_COLORS[int(stages[start])],
                       alpha=0.13, linewidth=0)
            start = i


def stage_legend(ax, **kw):
    handles = [Patch(color=c, alpha=0.35, label=f"{i} {n}") for i, (n, c) in enumerate(zip(STAGE_NAMES, STAGE_COLORS))]
    ax.legend(handles=handles, ncol=5, fontsize=8, **kw)


def describe(info):
    care = (f"watered below {info.water_below_pct:.0f} %" if pd.notna(info.water_below_pct) else "never watered")
    care += f", turned every {info.turn_every_days:.0f} days" if pd.notna(info.turn_every_days) else ", never turned"
    return f"{care}, heat_scale {info.heat_scale:.2f}"


tended = batch_info["train"].dropna(subset=["water_below_pct", "turn_every_days"]).iloc[0]
examples = [("train", tended, "Looked-after bin"),
            ("neglected", batch_info["neglected"].iloc[1], "Neglected bin"),
            ("small_basket", batch_info["small_basket"].iloc[2], "Small basket")]
fig, axes = plt.subplots(len(examples), 1, figsize=(11, 8), sharex=True)
for ax, (scenario, info, label) in zip(axes, examples):
    batch_id, title = info.batch_id, f"{label}: {describe(info)}"
    b = raw[scenario][raw[scenario].batch_id == batch_id]
    shade_stages(ax, b.elapsed_h, b.stage_id)
    ax.plot(b.elapsed_h / 24, b.temperature_c, color=INK, lw=0.9, label="Core temperature (°C)")
    ax.plot(b.elapsed_h / 24, b.ambient_c, color=MUTED, lw=0.6, label="Ambient (°C)")
    ax.plot(b.elapsed_h / 24, b.moisture_pct, color=SERIES[0], lw=1.0, ls="--", label="Moisture (%)")
    ax.set_title(title, loc="left")
    ax.set_ylim(15, 75)
axes[0].legend(loc="upper right", ncol=3, fontsize=8)
stage_legend(axes[-1], loc="upper center", bbox_to_anchor=(0.5, -0.28))
axes[-1].set_xlabel("Day of batch")
fig.tight_layout()
savefig(fig, "02_example_batches.png")
plt.show()

### 3.1 What a dried-out bin does in the simulator

Look at the neglected bin above. Nobody waters it, so after the hot phase moisture falls below
45 % (the emulator's `DRY_STALL`) and eventually to the 15 % floor. The emulator then slows the
stage clock: a pile below 30 % (`DRY_DEAD`) advances at only 5 % speed. That's a reasonable model
of microbes going dormant. The side effect is that the bin sits at **ambient temperature, bone dry,
still labelled "Cooling" (or "Maturation") for weeks**. Temperature can't tell those two apart,
so moisture ends up doing a lot of the work, and the main error in section 6 is Maturation
hours predicted as Cooling. The table shows how much of each stage's data comes from dry piles.

In [ ]:
DRY_STALL, DRY_DEAD = compost_model.DRY_STALL, compost_model.DRY_DEAD
r = raw["train"]
dry = r.assign(**{f"moisture < {DRY_STALL:.0f} % (stalling)": r.moisture_pct < DRY_STALL,
                  f"moisture < {DRY_DEAD:.0f} % (dormant)": r.moisture_pct < DRY_DEAD})
dry_table = dry.groupby("stage_id")[[c for c in dry.columns if c.startswith("moisture <")]].mean()
dry_table.index = [f"{i} {n}" for i, n in zip(STAGE_IDS, STAGE_NAMES)]
dry_table.style.format("{:.0%}")

In the literature a dry, dormant pile is a **process fault, not a composting stage**: Cornell and
LSU put the working range at 40–60 % moisture. The repo's mock data already has `dry_pile` and
`stalled_pile` fault codes (`simulator/mock_data.py`), so the conclusions recommend reporting this
as a fault flag next to the stage rather than folding it into "Cooling".

### 3.2 A check before modelling: do the simulated gases give the label away?

In `CompostModel.true_values()` the O₂, CO₂ and NH₃ values are computed from **the current
stage's own range**. Temperature carries over smoothly from one stage to the next, but the
gases jump the instant the stage changes. A model would learn that jump, which is the label
itself, and no real pile does that. The chart averages each signal around every stage change
in the training set.

In [ ]:
def around_transitions(frame, column, window_h=24):
    """Average a signal from window_h before to window_h after every stage change."""
    traces = {}
    for _, b in frame.groupby("batch_id"):
        b = b.set_index("elapsed_h")
        change_at = b.index[b.stage_id.diff().fillna(0) > 0]
        for t in change_at:
            seg = b.loc[t - window_h:t + window_h, column]
            traces.setdefault(int(b.loc[t, "stage_id"]), []).append(
                pd.Series(seg.values - seg.loc[:t].iloc[-12:-1].mean(), index=seg.index - t))
    return {s: pd.concat(v, axis=1).mean(axis=1).sort_index() for s, v in traces.items()}


fig, axes = plt.subplots(1, 3, figsize=(12, 3.2), sharex=True)
for ax, (column, label) in zip(axes, [("temperature_c", "Core temperature (°C)"),
                                     ("co2_pct", "CO₂ (%)"), ("nh3_relative", "NH₃ (relative)")]):
    for stage, trace in around_transitions(raw["train"], column).items():
        ax.plot(trace.index, trace.values, color=STAGE_COLORS[stage], label=f"into {stage} {STAGE_NAMES[stage]}")
    ax.axvline(0, color=MUTED, lw=0.8, ls=":")
    ax.set_title(label, loc="left")
    ax.set_xlabel("Hours from stage change")
axes[0].set_ylabel("Change vs. the 6 h before")
axes[1].legend(fontsize=7.5, loc="lower left")
fig.suptitle("Temperature changes smoothly at a stage boundary; the simulated gases jump", x=0.01, ha="left",
             fontweight="bold")
fig.tight_layout()
savefig(fig, "03_gas_leak_check.png")
plt.show()

The gas jumps happen at the exact moment the label changes, so any model given the simulated
O₂, CO₂, NH₃ or the MQ-135 value derived from them can read the label straight off the data.
We still score that feature set in section 5, flagged as leaky, but **the models we recommend
use only temperature, ambient and moisture**. Fixing this is a simulator change: compute gases
from a stage-independent curve of activity, or blend between stages. Until then the simulator
can't tell us whether a gas sensor is worth adding.

## 4. Feature engineering

Every feature uses **only past readings** (trailing windows, running maxima), so the server or
an ESP32 can compute it live. Missing readings are carried forward for up to 2 h, as a server
would. We then keep one row per hour for modelling.

| Group | Features | Why (section 1) |
|---|---|---|
| Level | `temp`, `temp_mean_6h`, `temp_mean_24h` | The phase bands are defined by temperature; the 24 h mean removes the day/night swing |
| Trend | `temp_slope_6h`, `temp_slope_24h` (°C/h), `temp_std_24h` | Separates heating from cooling at the same temperature (stage 1 vs 3) |
| Peak memory | `temp_peak`, `below_peak`, `hours_since_peak` | Cooling and curing are defined as *after* the peak |
| Sanitisation | `hours_above_45`, `hours_above_55` | Thermophilic onset, and the PFRP / EU time-at-temperature rules |
| Moisture | `moist`, `moist_mean_24h`, `moist_slope_24h` | Drying tracks thermophilic evaporation; 40–60 % optimum |
| Relative to ambient (°C) | `amb_mean_24h`, `dT`, `dT_mean_24h`, `dT_peak` | Maturity = back to ambient (needs the second probe) |
| Scale-free | `frac_of_peak`, `frac_of_peak_24h`, `slope_rel_6h`, `slope_rel_24h`, `std_rel_24h` | Heat above ambient as a fraction of *this bin's own* peak, so a 35 °C basket and a 65 °C pile look alike (section 1.3, point 2) |
| Gases (simulator only) | `o2`, `co2`, `nh3`, `mq135_mean_6h` | Leaky in this simulator (3.1) |
| Clock | `batch_day` | Known from when the bin was filled, but lets a model memorise the simulator's schedule |

In [ ]:
W6, W24 = int(6 / STEP_H), int(24 / STEP_H)


def make_features(frame):
    """Causal features per batch on a regular 30-minute grid, then one row per hour."""
    parts = []
    for batch_id, g in frame.groupby("batch_id", sort=False):
        grid = np.arange(0, g.elapsed_h.max() + STEP_H / 2, STEP_H)
        g = g.set_index("elapsed_h").reindex(grid)
        g[SENSOR_COLS] = g[SENSOR_COLS].ffill(limit=4)       # carry a reading forward for up to 2 h
        T, M, A = g.temperature_c, g.moisture_pct, g.ambient_c

        f = pd.DataFrame({"batch_id": batch_id, "elapsed_h": grid,
                          "stage_id": g.stage_id.ffill().bfill().astype(int).values})
        f.index = grid
        f["temp"] = T
        f["temp_mean_6h"] = T.rolling(W6, min_periods=1).mean()
        f["temp_mean_24h"] = T.rolling(W24, min_periods=1).mean()
        f["temp_slope_6h"] = f.temp_mean_6h.diff(W6).fillna(0) / 6
        f["temp_slope_24h"] = f.temp_mean_24h.diff(W24).fillna(0) / 24
        f["temp_std_24h"] = T.rolling(W24, min_periods=2).std().fillna(0)
        f["temp_peak"] = f.temp_mean_6h.cummax()
        f["below_peak"] = f.temp_peak - f.temp_mean_6h
        at_peak = pd.Series(np.where(f.temp_mean_6h >= f.temp_peak, grid, np.nan), index=grid).ffill()
        f["hours_since_peak"] = grid - at_peak
        f["hours_above_45"] = (f.temp_mean_6h > 45).cumsum() * STEP_H
        f["hours_above_55"] = (f.temp_mean_6h >= 55).cumsum() * STEP_H

        f["moist"] = M
        f["moist_mean_24h"] = M.rolling(W24, min_periods=1).mean()
        f["moist_slope_24h"] = f.moist_mean_24h.diff(W24).fillna(0) / 24

        f["amb_mean_24h"] = A.rolling(W24, min_periods=1).mean()
        f["dT"] = T - A
        f["dT_mean_24h"] = f.temp_mean_24h - f.amb_mean_24h
        dT_6h = f.temp_mean_6h - A.rolling(W6, min_periods=1).mean()
        f["dT_peak"] = dT_6h.cummax()
        peak_heat = f.dT_peak.clip(lower=1.0)
        f["frac_of_peak"] = dT_6h / peak_heat
        f["frac_of_peak_24h"] = f.dT_mean_24h / peak_heat
        f["slope_rel_6h"] = f.temp_slope_6h / peak_heat
        f["slope_rel_24h"] = f.temp_slope_24h / peak_heat
        f["std_rel_24h"] = f.temp_std_24h / peak_heat

        f["o2"], f["co2"], f["nh3"] = g.o2_pct, g.co2_pct, g.nh3_relative
        f["mq135_mean_6h"] = g.mq135.rolling(W6, min_periods=1).mean()
        f["batch_day"] = grid / 24
        parts.append(f[f.elapsed_h % 1 == 0])                 # one row per hour
    return pd.concat(parts, ignore_index=True)


t0 = time.time()
feats = {name: make_features(frame) for name, frame in raw.items()}
print(f"features in {time.time() - t0:.1f}s:", {k: len(v) for k, v in feats.items()})
train, test = feats["train"], feats["test"]
train.groupby("stage_id")[["temp_mean_24h", "temp_slope_24h", "hours_since_peak", "dT_mean_24h",
                           "frac_of_peak", "moist_mean_24h"]].median()

## 5. Evaluation protocol, and which sensors are worth having

**Split by batch.** Neighbouring hourly rows from the same bin are nearly identical. A random
row split puts near-copies of the test rows into training and flatters the score (section 1.3,
point 5). `GroupKFold` keeps each batch entirely in train or entirely in validation, and the
30 `test` batches are never touched until section 6.

**Metric: macro-F1.** It averages F1 over the five stages, so the short stages (0 and 1) count as
much as Maturation, which holds half the rows.

**Keeping it quick.** For comparisons we use a lighter gradient-boosting model on every 4th hour
of training data (neighbouring hours are near-duplicates anyway). The final models in section 6
are trained on every hour.

In [ ]:
TEMP_HISTORY = ["temp", "temp_mean_6h", "temp_mean_24h", "temp_slope_6h", "temp_slope_24h", "temp_std_24h",
                "temp_peak", "below_peak", "hours_since_peak", "hours_above_45", "hours_above_55"]
MOISTURE = ["moist", "moist_mean_24h", "moist_slope_24h"]
AMBIENT = ["amb_mean_24h", "dT", "dT_mean_24h", "dT_peak", "frac_of_peak", "slope_rel_6h"]
SCALE_FREE = ["frac_of_peak", "frac_of_peak_24h", "slope_rel_6h", "slope_rel_24h", "std_rel_24h",
              "hours_since_peak"] + MOISTURE
GASES = ["o2", "co2", "nh3", "mq135_mean_6h"]

FEATURE_SETS = {
    "Snapshot: temp + moisture now": ["temp", "moist"],
    "Core probe, trends only": ["temp", "temp_mean_6h", "temp_mean_24h", "temp_slope_6h", "temp_slope_24h", "temp_std_24h"],
    "Core probe + peak memory": TEMP_HISTORY,
    "+ moisture": TEMP_HISTORY + MOISTURE,
    "+ ambient probe": TEMP_HISTORY + MOISTURE + AMBIENT,
    "Scale-free (fraction of own peak)": SCALE_FREE,
    "Batch day only (schedule)": ["batch_day"],
    "+ simulated gases (leaky)": TEMP_HISTORY + MOISTURE + AMBIENT + GASES,
}


def hgb(**kw):
    """The full model, for the final fits."""
    return HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1, class_weight="balanced",
                                          random_state=SEED, **kw)


def quick_hgb():
    """A lighter model, for comparisons inside cross-validation."""
    return HistGradientBoostingClassifier(max_iter=60, learning_rate=0.2, max_leaf_nodes=15,
                                          class_weight="balanced", random_state=SEED)


cv_rows = train[train.elapsed_h % 4 == 0]


def grouped_cv(columns, data=cv_rows, folds=4):
    return cross_val_score(quick_hgb(), data[columns], data.stage_id, groups=data.batch_id,
                           cv=GroupKFold(folds), scoring="f1_macro")


# 1) how much does a random row split flatter the score?
rich = FEATURE_SETS["+ ambient probe"]
random_split = cross_val_score(quick_hgb(), cv_rows[rich], cv_rows.stage_id, scoring="f1_macro",
                               cv=KFold(4, shuffle=True, random_state=SEED)).mean()
print(f"100 batches  — random row split: {random_split:.3f}   split by batch: {grouped_cv(rich).mean():.3f}")

# 2) the pilot has 3 bins. Train on 3 batches and compare what random CV promises with what we get
pilot = train[train.batch_id.isin(train.batch_id.unique()[:3])]
promised = cross_val_score(quick_hgb(), pilot[rich], pilot.stage_id, scoring="f1_macro",
                           cv=KFold(4, shuffle=True, random_state=SEED)).mean()
delivered = f1_score(test.stage_id, quick_hgb().fit(pilot[rich], pilot.stage_id).predict(test[rich]), average="macro")
print(f"3 pilot bins — random row split promises {promised:.3f}; on 30 unseen batches it scores {delivered:.3f}")

In [ ]:
t0 = time.time()
cv_rows = []
for name, columns in FEATURE_SETS.items():
    scores = grouped_cv(columns)
    cv_rows.append({"feature set": name, "features": len(columns), "macro-F1": scores.mean(), "± (fold sd)": scores.std()})
cv_table = pd.DataFrame(cv_rows).set_index("feature set")
print(f"{len(FEATURE_SETS)} feature sets x 4 folds in {time.time() - t0:.0f}s")
cv_table

In [ ]:
order = cv_table.sort_values("macro-F1")
fig, ax = plt.subplots(figsize=(8.5, 3.6))
y = np.arange(len(order))
usable = np.array(["leaky" not in n and "schedule" not in n for n in order.index])
for keep, color, label in [(usable, SERIES[0], "usable on a real bin"), (~usable, MUTED, "not usable (leaky / schedule)")]:
    ax.errorbar(order["macro-F1"][keep], y[keep], xerr=order["± (fold sd)"][keep], fmt="o", ms=7, color=color,
                ecolor=color, elinewidth=1, capsize=0, label=label)
for yi, (v, sd) in enumerate(zip(order["macro-F1"], order["± (fold sd)"])):
    ax.text(v + sd + 0.006, yi, f"{v:.3f}", va="center", fontsize=8.5, color=INK_2)
ax.set_yticks(y, order.index)
ax.set_xlim(max(0.0, order["macro-F1"].min() - 0.05), 1.03)
ax.set_xlabel("Macro-F1, 4-fold split by batch (dot = mean, line = ± 1 sd across folds)")
ax.set_title("Which sensors and features earn their place?", loc="left")
ax.legend(loc="lower right", fontsize=8)
ax.grid(axis="y", visible=False)
savefig(fig, "04_feature_sets.png")
plt.show()

**Reading the comparison**

- **Moisture is the biggest single gain.** That's partly real (drying tracks the hot phase) and
  partly the dry-stall effect from 3.1.
- **Ambient adds a little on standard bins.** The emulator's core temperature barely depends on
  ambient, so most of its value only shows on unusual bins (section 9).
- **The scale-free set scores lower here** because it throws away absolute °C. It earns its
  place in section 9, where the bins are different.
- **The gas set's near-perfect score is the leak from 3.2,** not a real result.
- **`batch_day` alone gets a long way.** That shows how much of the label is simply the
  simulator's stage schedule.

The main models below use **"+ ambient probe"**: the core probe with peak memory, plus
moisture, plus a second DS18B20 for ambient. This is the best feature set we can actually
build. Section 9 compares it against the scale-free set.

In [ ]:
FEATS = FEATURE_SETS["+ ambient probe"]
X_train, y_train, g_train = train[FEATS], train.stage_id, train.batch_id
X_test, y_test = test[FEATS], test.stage_id
print(len(FEATS), "features:", ", ".join(FEATS))

## 6. Three classical models

1. **Literature rules (no training).** The thresholds come straight from section 1: thermophilic
   once the bin has passed 45 °C, the "peak" stage at ≥ 55 °C while still near its maximum,
   cooling once it's past the peak but more than 5 °C above ambient, and maturation when it's
   back near ambient.
2. **Decision tree.** It's tuned by batch-grouped CV, and we pick the *smallest* depth within
   0.02 F1 of the best so the tree stays readable and fits comfortably on the ESP32
   (exported in section 11).
3. **Gradient boosting** (`HistGradientBoostingClassifier`). The strongest tabular model; it
   runs on the server.

In [ ]:
def rule_based_stage(f):
    near_peak = f["below_peak"] < 3.0                       # still at (or close to) the hottest it has been
    return np.select(
        [f["temp_peak"] < 45,                                # never got past mesophilic yet
         near_peak & (f["temp_mean_6h"] < 55),               # climbing through the thermophilic band
         near_peak & (f["temp_mean_6h"] >= 55),              # sustained >= 55 °C (PFRP band)
         f["dT_mean_24h"] > 5.0],                            # past the peak, still clearly above ambient
        [0, 1, 2, 3], default=4)


def scores(y_true, y_pred):
    return {"macro-F1": f1_score(y_true, y_pred, average="macro"),
            "balanced acc.": balanced_accuracy_score(y_true, y_pred),
            "accuracy": accuracy_score(y_true, y_pred)}


results = {"Literature rules": scores(y_test, rule_based_stage(test))}

t0 = time.time()
tree_rows = train[train.elapsed_h % 2 == 0]
tree_search = GridSearchCV(
    DecisionTreeClassifier(class_weight="balanced", random_state=SEED),
    {"max_depth": [3, 4, 5, 6, 8, 10, 12], "min_samples_leaf": [25, 100]},
    cv=GroupKFold(4), scoring="f1_macro")
tree_search.fit(tree_rows[FEATS], tree_rows.stage_id, groups=tree_rows.batch_id)
grid = pd.DataFrame(tree_search.cv_results_)
best = grid.mean_test_score.max()
small_enough = grid[grid.mean_test_score >= best - 0.02].sort_values(["param_max_depth", "param_min_samples_leaf"],
                                                                      ascending=[True, False]).iloc[0]
tree = DecisionTreeClassifier(class_weight="balanced", random_state=SEED,
                              max_depth=int(small_enough.param_max_depth),
                              min_samples_leaf=int(small_enough.param_min_samples_leaf)).fit(X_train, y_train)
print(f"tree search {time.time() - t0:.0f}s — best CV {best:.3f}; chose depth {tree.get_depth()}, "
      f"{tree.get_n_leaves()} leaves (CV {small_enough.mean_test_score:.3f})")
results["Decision tree"] = scores(y_test, tree.predict(X_test))

by_depth = grid.groupby("param_max_depth").mean_test_score.max()
fig, ax = plt.subplots(figsize=(6.5, 2.6))
ax.plot(by_depth.index.astype(int), by_depth.values, color=SERIES[0], marker="o", ms=5)
ax.axvline(tree.get_depth(), color=MUTED, ls=":", lw=1)
ax.annotate(f"chosen: depth {tree.get_depth()}", (tree.get_depth(), by_depth.min()), xytext=(6, 0),
            textcoords="offset points", color=INK_2, fontsize=8.5)
ax.set_xlabel("max_depth")
ax.set_ylabel("CV macro-F1")
ax.set_title("We keep the shallowest tree within 0.02 of the best score", loc="left")
savefig(fig, "05a_tree_depth.png")
plt.show()

t0 = time.time()
booster = hgb().fit(X_train, y_train)
print(f"gradient boosting fit {time.time() - t0:.0f}s")
results["Gradient boosting"] = scores(y_test, booster.predict(X_test))

pd.DataFrame(results).T

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
for ax, (name, pred) in zip(axes, [("Literature rules", rule_based_stage(test)),
                                    ("Decision tree", tree.predict(X_test)),
                                    ("Gradient boosting", booster.predict(X_test))]):
    cm = confusion_matrix(y_test, pred, labels=STAGE_IDS, normalize="true")
    ConfusionMatrixDisplay(cm, display_labels=[f"{i}" for i in STAGE_IDS]).plot(
        ax=ax, cmap="Blues", colorbar=False, values_format=".2f")
    ax.set_title(f"{name} — macro-F1 {f1_score(y_test, pred, average='macro'):.3f}", loc="left")
    ax.set_xlabel("Predicted stage")
    ax.set_ylabel("True stage")
    ax.grid(False)
fig.text(0.01, -0.03, "Stages: " + "   ".join(f"{i} {n}" for i, n in zip(STAGE_IDS, STAGE_NAMES)),
         color=INK_2, fontsize=8.5)
fig.tight_layout()
savefig(fig, "05_confusion_matrices.png")
plt.show()

In [ ]:
sample = test.sample(min(len(test), 15_000), random_state=SEED)
perm = permutation_importance(booster, sample[FEATS], sample.stage_id, scoring="f1_macro",
                              n_repeats=5, random_state=SEED, n_jobs=-1)
importance = pd.Series(perm.importances_mean, index=FEATS).sort_values()

fig, ax = plt.subplots(figsize=(7.5, 5))
ax.barh(importance.index, importance.values, xerr=perm.importances_std[np.argsort(perm.importances_mean)],
        color=SERIES[0], height=0.6, error_kw={"ecolor": INK_2, "elinewidth": 0.8})
ax.set_xlabel("Drop in macro-F1 when the feature is shuffled (test set)")
ax.set_title("What gradient boosting relies on")
ax.grid(axis="y", visible=False)
savefig(fig, "06_permutation_importance.png")
plt.show()

## 7. Making predictions behave like a compost bin

A classifier scores each hour on its own, so its prediction can flicker between neighbouring
stages. A bin can't do that: stages only move forward (until fresh feedstock restarts the batch).
We add a small **causal hidden-Markov filter**:

- The only allowed moves are *stay* or *advance one stage*.
- The chance of advancing each hour is `1 / (typical stage length in hours)`, taken from the
  `duration_days` mode in `composting_stages.py`.
- The classifier's probabilities are used as the evidence.

The filter uses only the past, so it can run live on the server.

In [ ]:
STAGE_HOURS = np.array([s["duration_days"][1] * 24 for s in STAGES], dtype=float)
TRANSITIONS = np.zeros((len(STAGES), len(STAGES)))
for s in STAGE_IDS:
    advance = 0.0 if s == LAST_STAGE else 1.0 / STAGE_HOURS[s]
    TRANSITIONS[s, s] = 1.0 - advance
    if advance:
        TRANSITIONS[s, s + 1] = advance
def stage_filter(proba):
    """Forward (filtering) pass of a left-to-right HMM over one batch's hourly probabilities."""
    belief = np.eye(len(STAGES))[0]                          # a fresh batch starts in stage 0
    path = np.empty(len(proba), dtype=int)
    for i, p in enumerate(proba):
        # the models are trained with class_weight="balanced", so their probabilities already
        # act as likelihoods; an unweighted model would need dividing by the class prior here
        belief = (belief @ TRANSITIONS) * np.clip(p, 1e-4, None)
        belief /= belief.sum()
        path[i] = belief.argmax()
    return path


def filtered_predictions(frame, proba):
    out = np.empty(len(frame), dtype=int)
    for _, idx in frame.groupby("batch_id", sort=False).indices.items():
        out[idx] = stage_filter(proba[idx])
    return out


def flips_per_batch(frame, pred):
    s = pd.Series(pred, index=frame.index)
    return s.groupby(frame.batch_id).apply(lambda x: int((x.diff().fillna(0) != 0).sum())).mean()


def transition_lag(frame, pred):
    """Hours between each true stage change and the first prediction at (or past) the new stage."""
    lags = []
    for _, b in frame.assign(pred=pred).groupby("batch_id"):
        for stage in STAGE_IDS[1:]:
            true_t = b.loc[b.stage_id >= stage, "elapsed_h"]
            pred_t = b.loc[b.pred >= stage, "elapsed_h"]
            if len(true_t) and len(pred_t):
                lags.append({"into stage": stage, "lag_h": pred_t.iloc[0] - true_t.iloc[0]})
    return pd.DataFrame(lags)


proba_test = booster.predict_proba(X_test)
raw_pred = proba_test.argmax(axis=1)
smooth_pred = filtered_predictions(test, proba_test)
results["Gradient boosting + filter"] = scores(y_test, smooth_pred)

lag_raw, lag_smooth = transition_lag(test, raw_pred), transition_lag(test, smooth_pred)
lag_table = pd.DataFrame({
    "raw: median lag (h)": lag_raw.groupby("into stage").lag_h.median(),
    "filtered: median lag (h)": lag_smooth.groupby("into stage").lag_h.median(),
    "filtered: 90th pct (h)": lag_smooth.groupby("into stage").lag_h.quantile(0.9),
})
lag_table.index = [f"→ {s} {STAGE_NAMES[s]}" for s in lag_table.index]
print(f"stage changes per batch — truth: {flips_per_batch(test, y_test.values):.1f}, "
      f"raw: {flips_per_batch(test, raw_pred):.1f}, filtered: {flips_per_batch(test, smooth_pred):.1f}")
print("(negative lag = the model called the change early)")
display(pd.DataFrame(results).T.loc[["Gradient boosting", "Gradient boosting + filter"]])
lag_table

In [ ]:
batch_id = test.batch_id.unique()[4]
mask = (test.batch_id == batch_id).values
b = test[mask]
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 5.6), sharex=True, gridspec_kw={"height_ratios": [1.3, 1]})
shade_stages(ax1, b.elapsed_h, b.stage_id)
ax1.plot(b.elapsed_h / 24, b.temp, color=INK, lw=0.8, label="Core temperature (°C)")
ax1.plot(b.elapsed_h / 24, b.amb_mean_24h, color=MUTED, lw=1, label="Ambient, 24 h mean (°C)")
ax1.set_title(f"Test batch {batch_id}: true stage (background) vs. predictions", loc="left")
ax1.legend(loc="upper right", fontsize=8)
ax2.step(b.elapsed_h / 24, b.stage_id, where="post", color=INK, lw=2.2, label="True stage")
ax2.plot(b.elapsed_h / 24, raw_pred[mask], ".", ms=2.5, color=SERIES[1], label="Gradient boosting, raw")
ax2.step(b.elapsed_h / 24, smooth_pred[mask], where="post", color=SERIES[0], lw=1.4, label="With stage filter")
ax2.set_yticks(STAGE_IDS, [f"{i} {n}" for i, n in zip(STAGE_IDS, STAGE_NAMES)])
ax2.set_xlabel("Day of batch")
ax2.legend(loc="lower right", fontsize=8)
fig.tight_layout()
savefig(fig, "07_prediction_timeline.png")
plt.show()

## 8. A sequence model in PyTorch (GRU)

Instead of hand-built history features, a GRU reads the hourly readings in order and keeps its
own memory, so it can learn "has it peaked yet?" by itself. It sees only three raw inputs
(core temperature, moisture, ambient), so the comparison is with the same sensors as the
ambient-probe feature set. It is causal (one direction only), so it could run live. As noted in
section 1.3, we found no published GRU/LSTM for this task, so treat it as an experiment.

In [ ]:
import torch
from torch import nn

torch.manual_seed(SEED)
torch.set_num_threads(max(1, (os.cpu_count() or 2) // 2))
SEQ_COLS = ["temp", "moist", "amb_mean_24h"]
mu, sd = train[SEQ_COLS].mean(), train[SEQ_COLS].std()


def to_sequences(frame):
    xs, ys, index = [], [], []
    for _, idx in frame.groupby("batch_id", sort=False).indices.items():
        g = frame.iloc[idx]
        xs.append(torch.tensor(((g[SEQ_COLS] - mu) / sd).fillna(0.0).to_numpy(np.float32)))  # a missing first reading -> the mean
        ys.append(torch.tensor(g.stage_id.to_numpy(np.int64)))
        index.append(idx)
    return xs, ys, index


def padded(xs, ys):
    return (nn.utils.rnn.pad_sequence(xs, batch_first=True),
            nn.utils.rnn.pad_sequence(ys, batch_first=True, padding_value=-100))


class StageGRU(nn.Module):
    def __init__(self, n_inputs, hidden=64, n_stages=len(STAGES)):
        super().__init__()
        self.gru = nn.GRU(n_inputs, hidden, num_layers=2, batch_first=True, dropout=0.1)
        self.head = nn.Linear(hidden, n_stages)

    def forward(self, x):
        out, _ = self.gru(x)
        return self.head(out)


def gru_proba(net, frame):
    xs, _, index = to_sequences(frame)
    proba = np.zeros((len(frame), len(STAGES)), dtype=np.float32)
    net.eval()
    with torch.no_grad():
        for x, idx in zip(xs, index):
            proba[idx] = torch.softmax(net(x[None])[0], dim=-1).numpy()
    return proba


# hold back 15 training batches to choose the best epoch
batches = train.batch_id.unique()
val_batches = set(np.random.default_rng(SEED).choice(batches, 15, replace=False))
fit_frame = train[~train.batch_id.isin(val_batches)].reset_index(drop=True)
val_frame = train[train.batch_id.isin(val_batches)].reset_index(drop=True)
xs_fit, ys_fit, _ = to_sequences(fit_frame)

weights = torch.tensor(len(fit_frame) / (len(STAGES) * np.bincount(fit_frame.stage_id, minlength=len(STAGES))),
                       dtype=torch.float32)
net = StageGRU(len(SEQ_COLS), hidden=48)
optimiser = torch.optim.Adam(net.parameters(), lr=3e-3)
loss_fn = nn.CrossEntropyLoss(weight=weights, ignore_index=-100)
EPOCHS, BATCH = 20, 8

history, best_f1, best_state = [], -1.0, None
t0 = time.time()
for epoch in range(EPOCHS):
    net.train()
    order = torch.randperm(len(xs_fit))
    total = 0.0
    for start in range(0, len(order), BATCH):
        pick = order[start:start + BATCH]
        X, Y = padded([xs_fit[i] for i in pick], [ys_fit[i] for i in pick])
        loss = loss_fn(net(X).reshape(-1, len(STAGES)), Y.reshape(-1))
        optimiser.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(net.parameters(), 1.0)
        optimiser.step()
        total += loss.item() * len(pick)
    val_f1 = f1_score(val_frame.stage_id, gru_proba(net, val_frame).argmax(1), average="macro")
    history.append({"epoch": epoch + 1, "train loss": total / len(xs_fit), "val macro-F1": val_f1})
    if val_f1 > best_f1:
        best_f1, best_state = val_f1, {k: v.clone() for k, v in net.state_dict().items()}
net.load_state_dict(best_state)
print(f"GRU trained in {time.time() - t0:.0f}s; best validation macro-F1 {best_f1:.3f}")

history = pd.DataFrame(history)
fig, ax = plt.subplots(figsize=(7, 2.8))
ax.plot(history.epoch, history["val macro-F1"], color=SERIES[0], marker="o", ms=3)
ax.set_xlabel("Epoch")
ax.set_ylabel("Validation macro-F1")
ax.set_title("GRU validation score per epoch (the best epoch is kept)", loc="left")
savefig(fig, "08_gru_training.png")
plt.show()

proba_gru = gru_proba(net, test)
results["GRU (PyTorch)"] = scores(y_test, proba_gru.argmax(1))
results["GRU + filter"] = scores(y_test, filtered_predictions(test, proba_gru))
print(f"stage changes per batch — GRU raw: {flips_per_batch(test, proba_gru.argmax(1)):.1f}")
pd.DataFrame(results).T

The GRU carries its own memory, so compare its flip count with gradient boosting's in section 7.
The filter adds lag and costs the GRU accuracy rather than helping it, so the stress tests in
section 9 use the GRU on its own.

## 9. Stress tests: bins the models have never seen

The `test` set comes from the same distribution as training. Real bins won't, so we also score
the models on four shifted scenarios:

- **hot season:** ambient 30–34 °C
- **neglected:** never watered or turned, so the pile dries out and stalls
- **small basket:** a basket that only reaches 35–45 °C, as the Takakura studies in section 1 report
- **noisy sensors:** 2.5× the noise

We also try two ways of making the model robust to bin size:

- **Scale-free features** (section 4): describe the bin relative to its own peak, not in °C.
- **Varied bins:** train on 100 extra batches whose `heat_scale` is randomised between 0.2
  and 1.0. This is cheap *domain randomisation*, so the model sees hot and lukewarm bins alike.

In [ ]:
raw["train_varied"], batch_info["train_varied"] = simulate_many("train_varied", n=100, seed0=60_000,
                                                                heat_scale=(0.2, 1.0))
feats["train_varied"] = make_features(raw["train_varied"])
varied = feats["train_varied"]

t0 = time.time()
booster_scalefree = hgb().fit(train[SCALE_FREE], train.stage_id)
booster_varied = hgb().fit(varied[FEATS], varied.stage_id)
booster_robust = hgb().fit(varied[SCALE_FREE], varied.stage_id)
# the same robust recipe for the ESP32 tree, at the depth chosen in section 6
tree_robust = DecisionTreeClassifier(class_weight="balanced", random_state=SEED, max_depth=tree.get_depth(),
                                     min_samples_leaf=tree.min_samples_leaf).fit(varied[SCALE_FREE], varied.stage_id)
print(f"four extra models in {time.time() - t0:.0f}s")

MODELS = {
    "Literature rules": lambda f: rule_based_stage(f),
    "Decision tree": lambda f: tree.predict(f[FEATS]),
    "Decision tree, scale-free varied bins": lambda f: tree_robust.predict(f[SCALE_FREE]),
    "Gradient boosting + filter": lambda f: filtered_predictions(f, booster.predict_proba(f[FEATS])),
    "GRU": lambda f: gru_proba(net, f).argmax(1),
    "Varied bins, °C features + filter": lambda f: filtered_predictions(f, booster_varied.predict_proba(f[FEATS])),
    "Scale-free + filter": lambda f: filtered_predictions(f, booster_scalefree.predict_proba(f[SCALE_FREE])),
    "Scale-free, varied bins + filter": lambda f: filtered_predictions(f, booster_robust.predict_proba(f[SCALE_FREE])),
}
STRESS = ["test", "hot_season", "neglected", "small_basket", "noisy_sensors"]

t0 = time.time()
stress = pd.DataFrame({
    scenario: {name: f1_score(feats[scenario].stage_id, predict(feats[scenario].reset_index(drop=True)), average="macro")
               for name, predict in MODELS.items()}
    for scenario in STRESS
})
print(f"stress tests in {time.time() - t0:.0f}s (macro-F1)")
stress

In [ ]:
shown = ["Gradient boosting + filter", "GRU", "Scale-free + filter", "Scale-free, varied bins + filter"]
x = np.arange(len(STRESS))
width = 0.2
fig, ax = plt.subplots(figsize=(11, 3.8))
for i, name in enumerate(shown):
    ax.bar(x + (i - 1.5) * width, stress.loc[name, STRESS], width=width - 0.02, color=SERIES[i], label=name)
ax.set_xticks(x, [s.replace("_", " ") for s in STRESS])
ax.set_ylim(0, 1.05)
ax.set_ylabel("Macro-F1")
ax.set_title("Models that read absolute °C collapse on the small basket; scale-free features hold up", loc="left")
ax.legend(ncol=4, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.12))
ax.grid(axis="x", visible=False)
savefig(fig, "09_stress_tests.png")
plt.show()

## 10. Is "Peak Decomposition" a real class? A 4-stage variant

Section 1.3 noted that the literature doesn't treat Peak Decomposition as a separate phase. Here
we merge stages 1 and 2 into one **Thermophilic** class and re-score the same predictions. If the
score barely moves, the 1-vs-2 boundary isn't where the errors are. In the simulator stage 2 is
simply "hotter than about 57 °C", so models separate it easily. Real bins may not be so tidy.

In [ ]:
MERGE = np.array([0, 1, 1, 2, 3])          # 0 Mesophilic, 1 Thermophilic, 2 Cooling, 3 Maturation
four = {}
for name in ["Gradient boosting + filter", "GRU", "Scale-free, varied bins + filter"]:
    pred = MODELS[name](test.reset_index(drop=True))
    four[name] = {"5 stages": f1_score(y_test, pred, average="macro"),
                  "4 stages (1+2 merged)": f1_score(MERGE[y_test], MERGE[pred], average="macro")}
pd.DataFrame(four).T

## 11. Export for the device and the server

- **ESP32:** a decision tree as a plain C function (`compost_stage_tree.h`). We export the
  **scale-free, varied-bins tree** rather than the section 6 tree, because section 9 showed the
  °C-based tree collapsing on small baskets. To compute the features the firmware needs:
  - a 24 h ring buffer (48 readings at 30-minute intervals);
  - the running peak heat above ambient, and the hours since that peak;
  - the second DS18B20 for ambient.

  Subtrees whose leaves all give the same stage are collapsed, so the C code stays short.
- **Server (backendAPI / research pipeline):** both gradient boosting models with their feature
  lists and the filter's transition matrix (`compost_stage_model.joblib`): the "+ ambient probe"
  model and the robust scale-free, varied-bins model. Also the GRU weights (`compost_stage_gru.pt`).

In [ ]:
def tree_to_c(model, feature_names, function="compost_stage"):
    t = model.tree_
    lines = ["// Generated by ml/compost_stage_classifier.ipynb. Do not edit by hand.",
             "// Returns the compost stage id: " + ", ".join(f"{i} = {n}" for i, n in zip(STAGE_IDS, STAGE_NAMES)),
             "// Feature definitions: make_features() in section 4 of the notebook.",
             "// f[] holds the features in this order:"]
    lines += [f"//   f[{i:2d}] {name}" for i, name in enumerate(feature_names)]
    lines += ["#pragma once", "", f"static inline int {function}(const float *f) {{"]

    def stages_below(node):
        if t.children_left[node] == -1:
            return {int(np.argmax(t.value[node]))}
        return stages_below(t.children_left[node]) | stages_below(t.children_right[node])

    def walk(node, depth):
        pad = "    " * depth
        stages = stages_below(node)
        if len(stages) == 1:                  # a leaf, or a split that can't change the answer
            stage = stages.pop()
            lines.append(f"{pad}return {stage};  // {STAGE_NAMES[stage]}")
            return
        i = t.feature[node]
        lines.append(f"{pad}if (f[{i}] <= {t.threshold[node]:.4f}f) {{  // {feature_names[i]}")
        walk(t.children_left[node], depth + 1)
        lines.append(f"{pad}}} else {{")
        walk(t.children_right[node], depth + 1)
        lines.append(f"{pad}}}")

    walk(0, 1)
    lines.append("}")
    return "\n".join(lines) + "\n"


header = tree_to_c(tree_robust, SCALE_FREE)
(OUT / "compost_stage_tree.h").write_text(header)
(OUT / "compost_stage_tree_rules.txt").write_text(export_text(tree_robust, feature_names=SCALE_FREE, decimals=3))
print(f"compost_stage_tree.h: {header.count(chr(10))} lines, {tree_robust.get_n_leaves()} leaves before collapsing")
joblib.dump({"models": {"ambient_probe": {"model": booster, "features": FEATS},
                        "robust": {"model": booster_robust, "features": SCALE_FREE}},
             "stage_names": STAGE_NAMES, "step_hours": STEP_H, "transitions": TRANSITIONS},
            OUT / "compost_stage_model.joblib")
torch.save({"state_dict": net.state_dict(), "inputs": SEQ_COLS, "mean": mu.to_dict(), "std": sd.to_dict()},
           OUT / "compost_stage_gru.pt")

fig, ax = plt.subplots(figsize=(14, 5.5))
plot_tree(tree_robust, max_depth=2, feature_names=SCALE_FREE,
          class_names=[f"{i} {n}" for i, n in zip(STAGE_IDS, STAGE_NAMES)],
          filled=False, impurity=False, proportion=True, precision=2, fontsize=9, ax=ax)
ax.set_title(f"ESP32 decision tree (scale-free, varied bins): top 2 of {tree_robust.get_depth()} levels",
             loc="left")
savefig(fig, "10_decision_tree.png")
plt.show()
print(*sorted(p.name for p in OUT.iterdir()), sep="\n")

## Appendix: does the SmartBox model's data look like ours?

If the SmartBox Peltier CSV (`smartbox-ml-kompospeltier-data-fase.csv`) is present (set
`SMARTBOX_CSV`, or leave it in `~/Downloads`), we map its six phases onto our stages and run our
gradient boosting model on it. Their box is **actively heated** (a Peltier holds the core at
46–61 °C for 53 days) and humidified, so we expect our passive-bin model to fail on it. That
failure is the evidence that their trained model shouldn't be reused for our bins either.
Phase mapping: 1 Lag + 2 Mesophilic I → 0; 3 Thermophilic → 1; 4 Cooling → 3;
5 Maturation + 6 Stable → 4. We score it on 4 merged stages, because their thermophilic phase
covers both our stage 1 and stage 2.

In [ ]:
SMARTBOX_CSV = Path(os.environ.get("SMARTBOX_CSV", Path.home() / "Downloads" / "smartbox-ml-kompospeltier-data-fase.csv"))
if SMARTBOX_CSV.exists():
    sb = pd.read_csv(SMARTBOX_CSV).sort_values("step")
    sb_raw = pd.DataFrame({
        "batch_id": "smartbox", "elapsed_h": sb.elapsed_h, "temperature_c": sb.t_in,
        "moisture_pct": sb.moisture, "ambient_c": sb.t_amb, "o2_pct": np.nan, "co2_pct": np.nan,
        "nh3_relative": np.nan, "mq135": np.nan,
        "stage_id": sb.phase.map({1: 0, 2: 0, 3: 1, 4: 3, 5: 4, 6: 4}),
    })
    sb_feats = make_features(sb_raw)
    for label, model, columns in [("+ ambient probe", booster, FEATS), ("robust (scale-free, varied bins)", booster_robust, SCALE_FREE)]:
        sb_pred = filtered_predictions(sb_feats, model.predict_proba(sb_feats[columns]))
        print(f"{label:34s} 4-stage macro-F1 on their box: "
              f"{f1_score(MERGE[sb_feats.stage_id], MERGE[sb_pred], average='macro'):.3f}")
    display(pd.crosstab(pd.Series(sb_feats.stage_id.map(lambda s: STAGE_NAMES[s]), name="their phase → our stage"),
                        pd.Series(sb_pred, name="robust model's prediction")))
else:
    print(f"SmartBox CSV not found at {SMARTBOX_CSV}; skipping this appendix")

## 12. Conclusions and recommendations

The headline numbers are in the summary at the top. This section turns them into decisions.

### For the classifier
- **Server:** deploy the scale-free, varied-bins gradient boosting model with the stage filter (`compost_stage_model.joblib`, under `models["robust"]`). Compute its features per bin from the last 24 h of readings plus the running peak, using `make_features` in section 4.
- **GRU:** keep it as the research comparison. It is the best model on normal bins, but because it reads raw °C it shares the small-basket weakness (0.29). Feeding it scale-free inputs is the obvious next experiment.
- **ESP32:** treat the exported tree as an offline fallback. It scores 0.83 on normal bins, well below the server model, but it never collapses the way the °C tree does.
- **Fault flags:** report these next to the stage, not as a stage:
  - `dry_pile`: moisture below 40 % for 24 h (the literature's lower bound)
  - `too_wet`: above 60 %
  - `stalled_pile`: near ambient long before maturation is due

  The repo's mock data already defines `dry_pile` and `stalled_pile`.

### For the hardware
- **Second DS18B20 on the existing 1-Wire bus (GPIO17)** for ambient temperature. The robust features depend on it. A weather API is a fallback, but it measures outdoor air, not the air beside the bin.
- **Calibrate the capacitive moisture sensor** against weighed (gravimetric) samples of the actual feedstock. Calibration depends on texture and temperature, and the archived firmware's dry/wet constants are marked TODO.
- **Treat the MQ-135 as a qualitative trend only.** It needs a 48 h preheat and temperature/humidity compensation.

### For the simulator (`simulator/composting_stages.py`, `emulator/model.py`)
1. Compute gases from a stage-independent curve of activity, and keep O₂ + CO₂ near 21 %, so gas readings stop leaking the stage (3.2).
2. Merge Peak Decomposition into Active Thermophilic, or redefine it as "≥ 55 °C sustained" to match the literature and the EPA PFRP rule.
3. Add a bin-size or heat factor like `heat_scale` here. Indonesian Takakura baskets peak around 35 °C.
4. Randomise the starting moisture. Every batch starts at 62.5 %, so early on moisture works as a clock: the exported tree's first split is on `moist_mean_24h`. Real bins won't be that tidy.
5. Report a dried-out pile as a fault instead of freezing it in Cooling (3.1).
6. Give bokashi its own label set. It is fermentation, not thermophilic composting.

### For real data
- **Labels:** for each pilot batch, researchers record:
  - the date of peak temperature
  - when cooling starts
  - a maturity check: self-heating, Solvita, or the SNI 19-7030-2004 criteria the Indonesian Takakura studies use

  Without labels nothing can be trained or validated on real bins.
- **Evaluate by bin only.** With three bins, that means leave-one-bin-out. Expect a random split to overstate the score, as in section 5.

### Limitations
- Every label comes from our own simulator, so these scores measure how well a model recovers the simulator's stage rules, not compost biology.
- The ambient swing, core heat leakage, owner behaviour and small-basket scaling are assumptions layered on the emulator, not measurements.
- Each scenario is one fixed set of seeds. The fold-to-fold spread in section 5 (± 0.01–0.02) is a fair guide to the noise in these numbers.

## References

Sources marked † were checked only through the abstract or a search snippet.

- **Standards**
  - [US EPA 40 CFR 503, Appendix B](https://www.law.cornell.edu/cfr/text/40/appendix-B_to_part_503): PFRP / PSRP time-temperature rules
  - [EU 2019/1009, Annex II (CMC 3)](https://www.legislation.gov.uk/eur/2019/1009/annex/II/2020-01-31): compost sanitisation and stability
  - [NSW EPA 2016, garden organics](https://www.epa.nsw.gov.au/sites/default/files/rro16-garden-organics.pdf): references AS 4454-2012
- **Extension guidance**
  - [Cornell Waste Management Institute: compost microorganisms](https://compost.css.cornell.edu/microorg.html)
  - [Cornell FS4: oxygen](https://www.compost.css.cornell.edu/Factsheets/FS4.html)
  - [Cornell: monitoring moisture](https://www.compost.css.cornell.edu/monitor/monitormoisture.html)
  - [CCE Tompkins: compost testing](https://ccetompkins.org/resources/compost-testing-for-finished-compost)
  - [LSU AgCenter: basic principles of composting](https://agsci.oregonstate.edu/sites/agscid7/files/snic/basic-principles-of-composting-lsu.pdf)
- **Process and maturity**
  - [Solvita compost test manual](https://projects.sare.org/media/pdf/9/8/2/982665solvita-compost-test-manual.pdf)
  - [Cabañas-Vargas et al. 2005](https://www.revista.ingenieria.uady.mx/volumen9/assessing.pdf): windrow temperatures, Dewar self-heating
  - [Huang et al. 2017, PLOS ONE](https://pmc.ncbi.nlm.nih.gov/articles/PMC5467826/): maturity criteria
  - [Pagans et al. 2006, Chemosphere](https://ddd.uab.cat/pub/artpub/2006/163578/chemosphere_a2006m3v62n9p1534.pdf) †: NH₃ emission vs. temperature
  - [Jackson & Line 1998](https://eprints.utas.edu.au/11434/) †: windrow O₂ and the CO₂/O₂ quotient
  - [Lim et al. 2017, CET](https://www.aidic.it/cet/17/61/173.pdf): in-vessel curing, small loads
- **Takakura and bokashi**
  - [Nurita et al. 2023](https://ojs.poltekkesbengkulu.ac.id/index.php/sanpro/article/download/383/236): Takakura temperature profile, Indonesia
  - [Larasati 2019](https://ikesma.journal.unej.ac.id/article/view/14156/): Takakura temperatures
  - [Polinema IoT Takakura](https://jartel.polinema.ac.id/index.php/jartel/article/view/533) †
  - [Campos-Rodríguez 2016](https://revistas.tec.ac.cr/index.php/tec_marcha/article/view/2982)
  - [UniKL indoor composter](https://ir.unikl.edu.my/jspui/handle/123456789/7114) †
  - [FEMS Microbes 2026](https://doi.org/10.1093/femsmc/xtag018): bokashi microbiology
  - [Lew et al. 2021](https://pmc.ncbi.nlm.nih.gov/articles/PMC8073414/): smart bokashi bin
- **Sensors**
  - [Hanwei MQ-135 datasheet](https://cdn.shopify.com/s/files/1/2132/9029/files/MQ-135.pdf)
  - [Winsen MQ-135 manual](https://www.winsen-sensor.com/manual/mq135.html)
- **Machine learning and evaluation**
  - [Ma et al. 2025, CET](https://www.cetjournal.it/cet/25/122/004.pdf): GMM phase mapping + AutoML
  - [Wei et al. 2022](https://redi.cedia.edu.ec/document/529930) †: compost maturity regression
  - [Ding et al. 2022](https://agris.fao.org/search/en/records/65de2dba4c5aef494fdaa336) †
  - [Aydın Temel et al. 2023 review](https://avesis.marmara.edu.tr/yayin/ea694acd-5383-4689-8a80-3e7238935798/artificial-intelligence-and-machine-learning-approaches-in-composting-process-a-review) †
  - [Huang et al. 2024, Waste Management](https://doi.org/10.1016/j.wasman.2024.02.022) †
  - [Roberts et al. 2017, Ecography](https://www.biometrie.uni-freiburg.de/mitarbeiter/dormann/roberts-et-al-2017-ecography.pdf): blocked cross-validation
  - [Kapoor & Narayanan 2023](https://ar5iv.labs.arxiv.org/html/2207.07048): leakage in ML-based science